# CardioGraph - 12-lead ECG multi-label classification (PTB-XL, timm EfficientNet-B0)

Training pipeline for the CardioGraph ECG model served by the `ECG_Model` FastAPI service.

| | |
|---|---|
| **Task** | Multi-label classification of 52 SCP-ECG statements (rhythm, form, diagnostic) |
| **Data** | PTB-XL, 21,799 ten-second 12-lead ECGs, high-rate `records500` (500 Hz) |
| **Input** | 12 x 5000 signal, per-lead z-score, bilinear-resized to a 224 x 224 image, repeated to 3 channels |
| **Model** | `timm` `tf_efficientnet_b0`, ImageNet pretrained, 52 sigmoid outputs |
| **Split** | PTB-XL `strat_fold` 1-8 train, 9 validation, 10 test (patient-wise) |
| **Output** | `timm_efficientnet_b0_imagenet_ecg_multilabel_hr_500hz_best.pth` (drop-in for `ECG_Model/model/`) |

The checkpoint format, label order and preprocessing match `ECG_Model/inference.py` exactly, so a model trained
here loads in the API without code changes. The production checkpoint reached **validation macro AUC 0.893 at epoch 3**.

> Rebuilt pipeline (written by Claude) from the production checkpoint's saved config; the original Kaggle notebook was not kept.

**Setup on Kaggle:** add the PTB-XL dataset (PhysioNet `ptb-xl` 1.0.3) as input and turn on a GPU. The data folder is found automatically.

In [ ]:
!pip install -q timm wfdb

## Imports & config

In [ ]:
import os, ast, json, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
import wfdb
from scipy.signal import resample
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
from tqdm.auto import tqdm

# SMOKE_TEST=1 runs a tiny CPU pass to check the pipeline end to end (no pretrained download)
SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"


def find_data_root():
    if os.environ.get("PTBXL_ROOT"):
        return Path(os.environ["PTBXL_ROOT"])
    for base in [Path("/kaggle/input"), Path(".")]:
        if base.exists():
            hits = sorted(base.rglob("ptbxl_database.csv"))
            if hits:
                return hits[0].parent
    raise FileNotFoundError("ptbxl_database.csv not found - attach the PTB-XL dataset")


class CFG:
    DATA_ROOT = find_data_root()
    OUT_DIR = Path(os.environ.get("OUT_DIR", "/kaggle/working" if Path("/kaggle/working").exists() else "outputs"))

    MODEL_NAME = "tf_efficientnet_b0"
    PRETRAINED = not SMOKE_TEST

    SAMPLING_RATE = 500          # records500 (filename_hr)
    DURATION_S = 10
    SIGNAL_LENGTH = 5000         # 10 s x 500 Hz
    IMAGE_SIZE = 224

    TRAIN_FOLDS = list(range(1, 9))
    VAL_FOLDS = [9]
    TEST_FOLDS = [10]

    EPOCHS = 1 if SMOKE_TEST else 15
    BATCH_SIZE = 4 if SMOKE_TEST else 32
    LR = 3e-4
    WEIGHT_DECAY = 1e-4
    POS_WEIGHT_MAX = 30.0        # cap for rare-label positive weighting in BCE
    EARLY_STOP_PATIENCE = 4
    AUGMENT = True
    THRESHOLD = 0.5              # same default as the API (.env THRESHOLD)

    NUM_WORKERS = 0 if SMOKE_TEST else 2
    SEED = 42
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    CKPT_NAME = "timm_efficientnet_b0_imagenet_ecg_multilabel_hr_500hz_best.pth"


CFG.OUT_DIR.mkdir(parents=True, exist_ok=True)
USE_AMP = CFG.DEVICE == "cuda"


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


set_seed(CFG.SEED)
print("Data root:", CFG.DATA_ROOT)
print("Device   :", CFG.DEVICE, "| smoke test:", SMOKE_TEST)

## Labels

The 52 labels and their order are fixed to match the production checkpoint (`label_cols`). These are the PTB-XL
SCP statements with enough examples to learn; a record is positive for a label when the code appears in its `scp_codes`.

In [ ]:
LABEL_COLS = ["1AVB", "2AVB", "3AVB", "AFIB", "AFLT", "ALMI", "AMI", "ANEUR", "ASMI", "BIGU", "CLBBB", "CRBBB", "DIG", "EL", "ILBBB", "ILMI", "IMI", "INJAL", "INJAS", "INJIL", "INJIN", "INJLA", "IPLMI", "IPMI", "IRBBB", "ISCAL", "ISCAN", "ISCAS", "ISCIL", "ISCIN", "ISCLA", "ISC_", "IVCD", "LAFB", "LAO/LAE", "LMI", "LNGQT", "LPFB", "LVH", "NDT", "NORM", "NST_", "PAC", "PACE", "PMI", "PSVT", "PVC", "RAO/RAE", "RVH", "SEHYP", "STACH", "WPW"]
FIXED_COLS = ["ecg_id", "patient_id", "filename", "fold", "scp_labels", "active_codes"]
LABEL_INDEX = {c: i for i, c in enumerate(LABEL_COLS)}
print(len(LABEL_COLS), "labels")

In [ ]:
db = pd.read_csv(CFG.DATA_ROOT / "ptbxl_database.csv")
db["scp_codes"] = db["scp_codes"].apply(ast.literal_eval)

meta = pd.DataFrame({
    "ecg_id": db["ecg_id"].astype(int),
    "patient_id": db["patient_id"],
    "filename": db["filename_hr"],
    "fold": db["strat_fold"].astype(int),
    "scp_labels": db["scp_codes"].apply(lambda d: sorted(d.keys())),
})
meta["active_codes"] = meta["scp_labels"].apply(lambda ks: [k for k in ks if k in LABEL_INDEX])

Y = np.zeros((len(meta), len(LABEL_COLS)), dtype=np.float32)
for row, codes in enumerate(meta["active_codes"]):
    for code in codes:
        Y[row, LABEL_INDEX[code]] = 1.0

keep = Y.sum(axis=1) > 0
print(f"Records: {len(meta)} | with >=1 target label: {keep.sum()}")
meta = meta[keep].reset_index(drop=True)
Y = Y[keep]

# SCP code -> description (same file the API uses for its label mapping)
scp = pd.read_csv(CFG.DATA_ROOT / "scp_statements.csv", index_col=0)
DESCRIPTIONS = scp["description"].to_dict()

counts = pd.Series(Y.sum(0), index=LABEL_COLS).sort_values(ascending=False)
print(counts.head(10).to_string())
print("...")
print(counts.tail(5).to_string())

## Signal cache

Reading ~22k WFDB records every epoch is slow, so every record is converted once to a fixed 5000 x 12 array
(crop/pad to 10 s at its own sampling rate, then resample - same steps as the API) and stored as a float16 memmap.

In [ ]:
def ensure_samples_by_leads(signal):
    """(samples, 12) float32, same rules as ECG_Model/inference.py."""
    signal = np.asarray(signal)
    if signal.shape[0] == 12 and signal.shape[1] != 12:
        signal = signal.T
    if signal.shape[1] > 12:
        signal = signal[:, :12]
    if signal.shape[1] < 12:
        signal = np.pad(signal, ((0, 0), (0, 12 - signal.shape[1])))
    return signal.astype(np.float32)


def to_fixed_length(signal, fs):
    signal = np.nan_to_num(ensure_samples_by_leads(signal), nan=0.0, posinf=0.0, neginf=0.0)
    if fs:
        expected = int(float(fs) * CFG.DURATION_S)
        if signal.shape[0] > expected:
            signal = signal[:expected]
        elif signal.shape[0] < expected:
            signal = np.pad(signal, ((0, expected - signal.shape[0]), (0, 0)))
    if signal.shape[0] != CFG.SIGNAL_LENGTH:
        signal = resample(signal, CFG.SIGNAL_LENGTH, axis=0)
    return signal


cache_path = CFG.OUT_DIR / f"ptbxl_hr_{len(meta)}x{CFG.SIGNAL_LENGTH}x12_f16.npy"
if not cache_path.exists():
    tmp_path = cache_path.with_suffix(".tmp.npy")
    arr = np.lib.format.open_memmap(tmp_path, mode="w+", dtype=np.float16,
                                    shape=(len(meta), CFG.SIGNAL_LENGTH, 12))
    for i, fn in enumerate(tqdm(meta["filename"], desc="Caching signals")):
        sig, fields = wfdb.rdsamp(str(CFG.DATA_ROOT / fn))
        arr[i] = to_fixed_length(sig, fields.get("fs"))
    arr.flush()
    del arr
    tmp_path.rename(cache_path)

SIGNALS = np.load(cache_path, mmap_mode="r")
print("Signal cache:", cache_path, SIGNALS.shape)

## Dataset

`signal_to_image` is the training-side twin of `preprocess_signal_for_timm` in the API:
per-lead z-score -> (12, 5000) -> bilinear resize to 224 x 224 -> repeat to 3 channels.
Training augmentation (per-lead amplitude scaling + small Gaussian noise) is applied to the raw signal before normalisation.

In [ ]:
def signal_to_image(signal):
    signal = signal.astype(np.float32)
    mean = signal.mean(axis=0, keepdims=True)
    std = signal.std(axis=0, keepdims=True)
    std[std < 1e-8] = 1.0
    signal = np.nan_to_num((signal - mean) / std, nan=0.0, posinf=0.0, neginf=0.0)
    x = torch.from_numpy(np.ascontiguousarray(signal.T))[None, None]          # (1, 1, 12, 5000)
    x = F.interpolate(x, size=(CFG.IMAGE_SIZE, CFG.IMAGE_SIZE), mode="bilinear", align_corners=False)
    return torch.nan_to_num(x.repeat(1, 3, 1, 1)[0])                         # (3, 224, 224)


class PTBXLDataset(Dataset):
    def __init__(self, indices, train=False):
        self.indices = np.asarray(indices)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        signal = np.array(SIGNALS[idx], dtype=np.float32)
        if self.train and CFG.AUGMENT:
            signal = signal * np.random.uniform(0.9, 1.1, size=(1, 12)).astype(np.float32)
            signal = signal + np.random.normal(0, 0.01, size=signal.shape).astype(np.float32) * signal.std(0, keepdims=True)
        return signal_to_image(signal), torch.from_numpy(Y[idx])


folds = meta["fold"].values
train_idx = np.where(np.isin(folds, CFG.TRAIN_FOLDS))[0]
val_idx = np.where(np.isin(folds, CFG.VAL_FOLDS))[0]
test_idx = np.where(np.isin(folds, CFG.TEST_FOLDS))[0]
print(f"train {len(train_idx)} | val {len(val_idx)} | test {len(test_idx)}")

loader_kw = dict(batch_size=CFG.BATCH_SIZE, num_workers=CFG.NUM_WORKERS, pin_memory=CFG.DEVICE == "cuda")
train_loader = DataLoader(PTBXLDataset(train_idx, train=True), shuffle=True, drop_last=len(train_idx) > CFG.BATCH_SIZE, **loader_kw)
val_loader = DataLoader(PTBXLDataset(val_idx), shuffle=False, **loader_kw)
test_loader = DataLoader(PTBXLDataset(test_idx), shuffle=False, **loader_kw)

xb, yb = next(iter(train_loader))
print("Batch:", tuple(xb.shape), tuple(yb.shape))

## Model, loss, optimiser

In [ ]:
model = timm.create_model(CFG.MODEL_NAME, pretrained=CFG.PRETRAINED, num_classes=len(LABEL_COLS)).to(CFG.DEVICE)

pos = Y[train_idx].sum(0)
neg = len(train_idx) - pos
pos_weight = torch.tensor(np.clip(neg / np.maximum(pos, 1), 1.0, CFG.POS_WEIGHT_MAX), dtype=torch.float32)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(CFG.DEVICE))

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.EPOCHS)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
print(f"{CFG.MODEL_NAME}: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params")

In [ ]:
def macro_auc(y_true, y_prob):
    aucs = [roc_auc_score(y_true[:, j], y_prob[:, j])
            for j in range(y_true.shape[1]) if 0 < y_true[:, j].sum() < len(y_true)]
    return float(np.mean(aucs)) if aucs else float("nan")


def run_epoch(loader, train):
    model.train(train)
    total_loss, probs, targets = 0.0, [], []
    for x, y in tqdm(loader, desc="train" if train else "eval", leave=False):
        x, y = x.to(CFG.DEVICE, non_blocking=True), y.to(CFG.DEVICE, non_blocking=True)
        with torch.set_grad_enabled(train), torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            logits = model(x)
            loss = criterion(logits.float(), y)
        if train:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        total_loss += loss.item() * len(x)
        probs.append(torch.sigmoid(logits.float()).detach().cpu().numpy())
        targets.append(y.cpu().numpy())
    probs, targets = np.concatenate(probs), np.concatenate(targets)
    return total_loss / len(loader.dataset), macro_auc(targets, probs), probs, targets

## Train

In [ ]:
ckpt_path = CFG.OUT_DIR / CFG.CKPT_NAME
best_score, best_epoch, patience = -np.inf, 0, 0
history = []

for epoch in range(1, CFG.EPOCHS + 1):
    t0 = time.time()
    train_loss, train_auc, _, _ = run_epoch(train_loader, train=True)
    val_loss, val_auc, _, _ = run_epoch(val_loader, train=False)
    scheduler.step()
    history.append(dict(epoch=epoch, train_loss=train_loss, train_auc=train_auc, val_loss=val_loss, val_auc=val_auc))
    print(f"Epoch {epoch:02d} | train loss {train_loss:.4f} auc {train_auc:.4f} | "
          f"val loss {val_loss:.4f} auc {val_auc:.4f} | {time.time() - t0:.0f}s")

    score = val_auc if not np.isnan(val_auc) else -val_loss
    if score > best_score:
        best_score, best_epoch, patience = score, epoch, 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "epoch": epoch,
            "best_val_loss": val_loss,
            "best_val_auc": val_auc,
            "label_cols": LABEL_COLS,
            "fixed_cols": FIXED_COLS,
            "model_name": CFG.MODEL_NAME,
            "pretrained": CFG.PRETRAINED,
            "image_size": CFG.IMAGE_SIZE,
            "signal_length": CFG.SIGNAL_LENGTH,
            "config": {
                "signal_length": CFG.SIGNAL_LENGTH,
                "image_size": CFG.IMAGE_SIZE,
                "n_classes": len(LABEL_COLS),
                "model": CFG.MODEL_NAME,
                "pretrained_source": "timm ImageNet",
                "input_ecg": "HR 500Hz records500",
            },
        }, ckpt_path)
        print(f"  saved best -> {ckpt_path.name}")
    else:
        patience += 1
        if patience >= CFG.EARLY_STOP_PATIENCE:
            print("Early stopping")
            break

pd.DataFrame(history).to_csv(CFG.OUT_DIR / "ecg_training_history.csv", index=False)
print(f"Best epoch {best_epoch}")

## Test set (fold 10)

In [ ]:
ckpt = torch.load(ckpt_path, map_location=CFG.DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
test_loss, test_auc, test_prob, test_true = run_epoch(test_loader, train=False)
test_pred = (test_prob >= CFG.THRESHOLD).astype(int)

rows = []
for j, label in enumerate(LABEL_COLS):
    n_pos = int(test_true[:, j].sum())
    both = 0 < n_pos < len(test_true)
    rows.append({
        "label": label,
        "description": DESCRIPTIONS.get(label, label),
        "test_positives": n_pos,
        "auc": roc_auc_score(test_true[:, j], test_prob[:, j]) if both else np.nan,
        "average_precision": average_precision_score(test_true[:, j], test_prob[:, j]) if n_pos else np.nan,
        "f1@thr": f1_score(test_true[:, j], test_pred[:, j], zero_division=0),
    })
per_label = pd.DataFrame(rows).sort_values("test_positives", ascending=False)
per_label.to_csv(CFG.OUT_DIR / "ecg_test_per_label_metrics.csv", index=False)

print(f"Test loss      : {test_loss:.4f}")
print(f"Test macro AUC : {test_auc:.4f}")
print(f"Test macro F1  : {f1_score(test_true, test_pred, average='macro', zero_division=0):.4f} (threshold {CFG.THRESHOLD})")
print(f"Test micro F1  : {f1_score(test_true, test_pred, average='micro', zero_division=0):.4f}")
per_label.head(15)

## API compatibility check

Loads the checkpoint exactly the way `ECG_Model/inference.py::load_timm_model` does and predicts one test record.
If this cell runs, copy the `.pth` into `ECG_Model/model/` to deploy it.

In [ ]:
ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
api_model = timm.create_model(ck["model_name"], pretrained=False, num_classes=len(ck["label_cols"]))
api_model.load_state_dict(ck["model_state_dict"])   # strict, like the API
api_model.eval()

idx = test_idx[0]
sig, fields = wfdb.rdsamp(str(CFG.DATA_ROOT / meta.loc[idx, "filename"]))
x = signal_to_image(to_fixed_length(sig, fields.get("fs")))[None]
with torch.no_grad():
    probs = torch.sigmoid(api_model(x)).numpy()[0]

print("Record     :", meta.loc[idx, "filename"])
print("True codes :", meta.loc[idx, "active_codes"])
for j in np.argsort(-probs)[:5]:
    print(f"  {LABEL_COLS[j]:8s} {probs[j]:.3f}  {DESCRIPTIONS.get(LABEL_COLS[j], '')}")